# 01: Data Assembly

Loads the master and aux files, joins them and keeps the configured folds.

**Outputs:** data/01_assembled.parquet

In [ ]:
# Parameters (injected by papermill)
config_path = "config/car_coll/v1"

In [ ]:
import pandas as pd
import numpy as np
import yaml
import os
import sys
import gc
from datetime import datetime
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("Stage 01: Data assembly")

project_root = setup_notebook_environment()
print(f"\nProject root: {project_root}")

In [ ]:
# Load config
from utils import load_config
config_file = f"{config_path}/config.yaml"
print(f"\nLoading config: {config_file}")

cfg = load_config(config_file)

print(f"  Experiment: {cfg['experiment']['name']}")
print(f"  Target: {cfg['experiment']['target']}")
print(f"  Run mode: {cfg['run_mode']}")

In [ ]:
# Paths for this machine
pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'

master_data_path = cfg['machines'][pc_id]['paths']['master_data_path']
aux_data_path = cfg['machines'][pc_id]['paths']['aux_data_path']
output_base = cfg['machines'][pc_id]['paths']['output_path']
os.makedirs(f"{output_base}/data", exist_ok=True)

master_path = f"{master_data_path}/{cfg['data']['master_file']}"
aux_path = f"{aux_data_path}/{cfg['data']['aux_file']}"

print(f"\nData paths:")
print(f"  Master: {master_path}")
print(f"  Aux: {aux_path}")
print(f"  Output: {output_base}")

In [ ]:
# Determine which folds to load based on run mode
mode = cfg['run_mode']
test_folds = cfg[mode]['test_folds']
train_folds = cfg[mode]['train_folds']

print(f"\nFold configuration:")
print(f"  Run mode: {mode}")
print(f"  Train folds: {train_folds}")
print(f"  Test folds: {test_folds}")

In [ ]:
# Load auxiliary file FIRST (smaller, has fold column)
print(f"\nLoading auxiliary file...")
aux_data = pd.read_parquet(aux_path)
print(f"  Shape: {aux_data.shape}")
print(f"  Memory: {aux_data.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"  Columns: {aux_data.columns.tolist()[:10]}...")

# Check for fold column
fold_col = cfg['data']['fold_column']
if fold_col in aux_data.columns:
    print(f"  Fold column found: {fold_col}")
    print(f"  Fold values: {sorted(aux_data[fold_col].unique().tolist())}")
else:
    print(f"  ERROR: Fold column '{fold_col}' not found in aux data")
    print(f"  Available columns: {aux_data.columns.tolist()}")

In [ ]:
# Filter aux data to only needed folds (reduce memory before join)
needed_folds = train_folds + test_folds
aux_filtered = aux_data[aux_data[fold_col].isin(needed_folds)].copy()
print(f"\nFiltered aux data to folds {needed_folds}")
print(f"  Shape: {aux_filtered.shape}")
print(f"  Memory saved: {(aux_data.memory_usage(deep=True).sum() - aux_filtered.memory_usage(deep=True).sum()) / 1e9:.2f} GB")

del aux_data
gc.collect()

In [ ]:
# Find join key in aux data
join_key_config = cfg['data']['join_key']
print(f"\nLooking for join key '{join_key_config}'...")

possible_keys = [join_key_config, 'VIN_Date', 'vindate', 'vin_date', 'VINDATE']
join_key = None

for key in possible_keys:
    if key in aux_filtered.columns:
        join_key = key
        print(f"  Found join key: {join_key}")
        break

if join_key is None:
    print(f"  ERROR: Join key not found in aux data")
    print(f"  Aux columns: {aux_filtered.columns.tolist()[:20]}")

In [ ]:
# Load column subset (for performance)
columns_file = cfg['data'].get('columns_to_load_file')

if columns_file:
    columns_path = f"{config_path}/{columns_file}"
    if os.path.exists(columns_path):
        print(f"\nLoading column subset from: {columns_file}")
        cols_df = pd.read_csv(columns_path, comment='#')
        columns_to_load = cols_df['column_name'].tolist()
        if join_key not in columns_to_load:
            columns_to_load.append(join_key)
            print(f"  Added join key '{join_key}' to column list")
        print(f"  Columns requested: {len(columns_to_load)}")
    else:
        print(f"\nWARNING: Column file not found: {columns_path}")
        print(f"  Loading ALL columns (slower)")
        print(f"  Run Template 00 (setup) first to create column subset file")
        columns_to_load = None
else:
    print(f"\nNo column subset specified, loading ALL columns")
    columns_to_load = None

# Load master file with column filtering
print(f"\nLoading master file...")
if columns_to_load:
    # Targets (pp_bi etc.) are in the aux file, so keep only columns master has
    import pyarrow.parquet as pq
    master_schema = pq.read_schema(master_path)
    available_cols = set(master_schema.names)
    
    filtered_cols = [c for c in columns_to_load if c in available_cols]
    missing_cols = set(columns_to_load) - set(filtered_cols)
    
    if missing_cols:
        print(f"  Skipping {len(missing_cols)} columns not in master file:")
        for col in sorted(list(missing_cols)[:5]):
            print(f"    - {col}")
        if len(missing_cols) > 5:
            print(f"    ... and {len(missing_cols) - 5} more")
        print(f"    (These will be added from aux file during join)")
    
    print(f"  Loading {len(filtered_cols)} columns (optimized)")
    master_data = pd.read_parquet(master_path, columns=filtered_cols)
else:
    print(f"  Loading ALL columns (slower)")
    master_data = pd.read_parquet(master_path)

print(f"  Shape: {master_data.shape}")
print(f"  Memory: {master_data.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"  Columns (first 10): {master_data.columns.tolist()[:10]}")

In [ ]:
# Check if join key exists in master
if join_key not in master_data.columns:
    print(f"\nJoin key '{join_key}' not in master data")
    print(f"Trying alternative names...")
    
    for key in possible_keys:
        if key in master_data.columns:
            print(f"  Found '{key}' in master data")
            master_data = master_data.rename(columns={key: join_key})
            print(f"  Renamed '{key}' -> '{join_key}'")
            break

In [ ]:
# Perform join
print(f"\nJoining master and aux data...")
print(f"  Join key: {join_key}")
print(f"  Join type: {cfg['data']['join_type']}")

# Columns loaded from master that aux also has (vin, st_raw): keep master's, as 08 scoring does.
# Without this the join makes vin_x/vin_y, st_raw_x/st_raw_y.
dup = [c for c in aux_filtered.columns if c in master_data.columns and c != join_key]
if dup:
    print(f"  Using master values for: {dup}")
    aux_filtered = aux_filtered.drop(columns=dup)

data = master_data.merge(
    aux_filtered,
    on=join_key,
    how=cfg['data']['join_type']
)

print(f"  Joined shape: {data.shape}")
print(f"  Memory: {data.memory_usage(deep=True).sum() / 1e9:.2f} GB")

del master_data, aux_filtered
gc.collect()

In [ ]:
# Verify fold column exists
if fold_col in data.columns:
    print(f"\nFold column verified in merged data")
    print(f"  Fold counts:")
    fold_counts = data[fold_col].value_counts().sort_index()
    for fold, count in fold_counts.items():
        if fold in train_folds:
            print(f"    Fold {fold}: {count:,} rows (TRAIN)")
        elif fold in test_folds:
            print(f"    Fold {fold}: {count:,} rows (TEST)")
        else:
            print(f"    Fold {fold}: {count:,} rows (other)")
else:
    print(f"  ERROR: Fold column missing after join!")

In [ ]:
# Build combined target/exposure if config has experiment.combine (e.g. liab BI+PD); no-op otherwise
from utils import add_combined_columns, validate_critical_columns
data = add_combined_columns(data, cfg)
validate_critical_columns(data, cfg, 'Stage 01 Output')

# Save checkpoint
checkpoint_path = f"{output_base}/data/01_assembled.parquet"
print(f"\nSaving checkpoint...")
from utils import save_parquet
save_parquet(data, checkpoint_path)

file_size = os.path.getsize(checkpoint_path) / 1e9
print(f"  Saved: {checkpoint_path}")
print(f"  Size: {file_size:.2f} GB")
print(f"  Shape: {data.shape}")

In [ ]:
print(f"\nCleaning memory...")
del data
gc.collect()
print(f"  Memory cleaned")

In [ ]:
print("Stage 01 done")